# 04 · Ideação da Solução — Hit Maker (Spotify)

Módulo anterior: [03 · Tratamento e Limpeza dos Dados](03_tratamento_limpeza_dados.ipynb). Próximo módulo: [05 · Protótipo do Produto](05_prototipo_produto.ipynb).

Implementa as **Frentes 2 e 3** das GQs refinadas: primeiro a anatomia dos hits (EDA comparativa), depois a modelagem preditiva propriamente dita — é aqui que a "ideia" (existe uma fórmula estrutural para um hit?) vira uma resposta testável. Fecha com clustering, que complementa o modelo supervisionado descrevendo arquétipos sonoros.

Lê `artifacts/df_limpo.csv`, produzido no módulo 03.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, roc_auc_score, confusion_matrix,
    RocCurveDisplay,
)
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
RANDOM_STATE = 42

In [ ]:
COLUNAS = {
    "genero": "track_genre",
    "popularidade": "popularity",
    "id_faixa": "track_id",
    "nome_faixa": "track_name",
    "artista": "artists",
}

FEATURES_NUMERICAS = [
    "danceability", "energy", "valence", "tempo", "loudness",
    "acousticness", "instrumentalness", "liveness", "speechiness",
]

ARTIFACTS_DIR = Path("artifacts")
df_limpo = pd.read_csv(ARTIFACTS_DIR / "df_limpo.csv")
df_limpo.shape

## Frente 2: Anatomia e Fatores de Sucesso (EDA)

GQ: *Quais características diferenciam faixas de alta popularidade das demais? Quais gêneros têm maior densidade de hits vs. volume de lançamentos?*

Primeiro passo: transformar `popularity` (contínua) em um rótulo binário `is_hit`. **Esta é a decisão mais crítica do projeto** (decisão de negócio nº 1 do módulo 02) — usamos aqui o percentil 80 global como ponto de partida, com a alternativa por gênero já disponível.

In [ ]:
def definir_hit(df: pd.DataFrame, percentil_corte: float = 80.0):
    """
    Cria o rótulo binário 'is_hit'.
    # ATENÇÃO: este threshold é a decisão mais crítica do projeto — define
    # o que conta como "sucesso". Um percentil fixo global pode distorcer
    # gêneros de nicho (que nunca atingem popularidade alta em termos
    # absolutos). Considere threshold POR GÊNERO como alternativa (ver
    # `definir_hit_por_genero` abaixo).
    """
    df = df.copy()
    limiar = np.percentile(df[COLUNAS["popularidade"]], percentil_corte)
    df["is_hit"] = (df[COLUNAS["popularidade"]] >= limiar).astype(int)
    return df, limiar

In [ ]:
def definir_hit_por_genero(df: pd.DataFrame, percentil_corte: float = 80.0):
    """Threshold relativo ao próprio gênero — evita viés contra nichos."""
    df = df.copy()
    limiares = df.groupby(COLUNAS["genero"])[COLUNAS["popularidade"]].transform(
        lambda s: np.percentile(s, percentil_corte)
    )
    df["is_hit"] = (df[COLUNAS["popularidade"]] >= limiares).astype(int)
    return df

In [ ]:
df_limpo, limiar_hit = definir_hit(df_limpo, percentil_corte=80)
print("Limiar de popularidade para 'hit':", limiar_hit)
print("Proporção de hits:", df_limpo["is_hit"].mean())

In [ ]:
def comparar_hits_vs_demais(df: pd.DataFrame):
    """
    GQ: quais características diferenciam faixas de alta popularidade das demais?
    """
    comparacao = df.groupby("is_hit")[FEATURES_NUMERICAS].mean().T
    comparacao.columns = ["Não-hit", "Hit"]
    comparacao["diferenca_%"] = (
        (comparacao["Hit"] - comparacao["Não-hit"]) / comparacao["Não-hit"] * 100
    )
    return comparacao.sort_values("diferenca_%", ascending=False)


comparar_hits_vs_demais(df_limpo)

In [ ]:
def densidade_e_volatilidade_por_genero(df: pd.DataFrame):
    """
    GQ: quais gêneros têm maior densidade de hits vs. volume de lançamentos,
    e quais têm maior volatilidade de popularidade?
    """
    agrupado = df.groupby(COLUNAS["genero"]).agg(
        n_faixas=(COLUNAS["popularidade"], "size"),
        n_hits=("is_hit", "sum"),
        popularidade_media=(COLUNAS["popularidade"], "mean"),
        popularidade_desvio=(COLUNAS["popularidade"], "std"),
    )
    agrupado["densidade_de_hits"] = agrupado["n_hits"] / agrupado["n_faixas"]
    return agrupado.sort_values("densidade_de_hits", ascending=False)


densidade_e_volatilidade_por_genero(df_limpo).head(10)

## Frente 3: Modelagem Preditiva

GQ: *É possível treinar um modelo acurado para estimar a probabilidade de uma faixa atingir alta popularidade? Quais atributos exercem maior poder preditivo?*

In [ ]:
def preparar_features(df: pd.DataFrame, incluir_genero: bool = True):
    """Monta a matriz de features (X) e o alvo (y) para o modelo."""
    X = df[FEATURES_NUMERICAS].copy()
    if incluir_genero:
        # ATENÇÃO: one-hot em gêneros com alta cardinalidade pode gerar
        # muitas colunas esparsas. Avalie target/frequency encoding se
        # o dataset tiver dezenas de gêneros.
        genero_dummies = pd.get_dummies(df[COLUNAS["genero"]], prefix="genero")
        X = pd.concat([X, genero_dummies], axis=1)
    y = df["is_hit"]
    return X, y

In [ ]:
def treinar_modelo(X: pd.DataFrame, y: pd.Series):
    """
    GQ: 'É possível treinar um modelo acurado para estimar a probabilidade
    de uma faixa atingir alta popularidade?'
    """
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
    )

    # ATENÇÃO: o rótulo "hit" (top 20% por padrão) tende a gerar
    # desbalanceamento de classes (~80/20). Usar class_weight='balanced'
    # é o mínimo — considere também SMOTE se a performance for baixa.
    modelo = RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    modelo.fit(X_train, y_train)
    return modelo, X_train, X_test, y_train, y_test

In [ ]:
X, y = preparar_features(df_limpo)
modelo, X_train, X_test, y_train, y_test = treinar_modelo(X, y)
X.shape, y.mean()

In [ ]:
def avaliar_modelo(modelo, X_test, y_test):
    y_pred = modelo.predict(X_test)
    y_proba = modelo.predict_proba(X_test)[:, 1]

    print(classification_report(y_test, y_pred))
    print("ROC-AUC:", roc_auc_score(y_test, y_proba))

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt="d",
                cmap="Blues", ax=axes[0])
    axes[0].set_title("Matriz de Confusão")
    RocCurveDisplay.from_predictions(y_test, y_proba, ax=axes[1])
    axes[1].set_title("Curva ROC")
    plt.tight_layout()
    return y_pred, y_proba


y_pred, y_proba = avaliar_modelo(modelo, X_test, y_test)

In [ ]:
def importancia_features(modelo, colunas_X):
    """
    GQ: 'Quais atributos exercem maior poder preditivo?'
    """
    importancias = pd.Series(
        modelo.feature_importances_, index=colunas_X
    ).sort_values(ascending=False)
    plt.figure(figsize=(8, 6))
    importancias.head(15).plot(kind="barh")
    plt.gca().invert_yaxis()
    plt.title("Top 15 features mais preditivas de sucesso")
    plt.tight_layout()
    return importancias


ranking_features = importancia_features(modelo, X.columns)
ranking_features.head(15)

## Clustering e Similaridade Estrutural

Complementa a modelagem preditiva: em vez de só prever "probabilidade de hit", agrupa as faixas por perfil sonoro para responder diretamente "existe uma combinação de características associada às músicas de alta popularidade?" — cada cluster vira uma "assinatura sonora" com uma taxa de hits associada. Também é a base do módulo 05 (faixas subestimadas por similaridade estrutural, não só por probabilidade do modelo).

Decisão de negócio nº 10 do módulo 02: o `k` do KMeans deve ser escolhido olhando o gráfico de cotovelo/silhouette abaixo, não fixado sem validar.

In [ ]:
def escalar_features(df: pd.DataFrame, colunas: list = None):
    """
    Padroniza as features numéricas (média 0, desvio 1).
    # ATENÇÃO: clustering e KNN são sensíveis a escala — nunca rode sobre
    # os dados crus (ex: 'tempo' em BPM vs 'valence' entre 0-1 dominaria a
    # distância euclidiana sem normalização).
    """
    colunas = colunas or FEATURES_NUMERICAS
    scaler = StandardScaler()
    X_escalado = scaler.fit_transform(df[colunas])
    return pd.DataFrame(X_escalado, columns=colunas, index=df.index), scaler

In [ ]:
def encontrar_k_ideal(X_escalado: pd.DataFrame, k_min: int = 2, k_max: int = 12):
    """
    Testa vários valores de k e retorna inércia + silhouette score de cada um,
    para apoiar a escolha do número de clusters (método do cotovelo).
    # ATENÇÃO: rodar por gênero costuma fazer mais sentido do que no
    # dataset inteiro — "energy" e "tempo" médios variam muito entre gêneros,
    # então clusters globais tendem a só reaprender o próprio gênero.
    """
    resultados = []
    for k in range(k_min, k_max + 1):
        km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
        labels = km.fit_predict(X_escalado)
        resultados.append({
            "k": k,
            "inercia": km.inertia_,
            "silhouette": silhouette_score(X_escalado, labels),
        })
    resultado_df = pd.DataFrame(resultados)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(resultado_df["k"], resultado_df["inercia"], marker="o")
    axes[0].set_title("Método do cotovelo (inércia)")
    axes[0].set_xlabel("k")
    axes[1].plot(resultado_df["k"], resultado_df["silhouette"], marker="o", color="orange")
    axes[1].set_title("Silhouette score")
    axes[1].set_xlabel("k")
    plt.tight_layout()
    return resultado_df


resultado_k = encontrar_k_ideal(escalar_features(df_limpo)[0], k_min=2, k_max=10)
resultado_k

In [ ]:
def clusterizar_faixas(df: pd.DataFrame, k: int, colunas: list = None):
    """
    Roda o KMeans final e devolve o df com a coluna 'cluster', além do
    modelo e do scaler (necessários depois para posicionar novas faixas).
    """
    colunas = colunas or FEATURES_NUMERICAS
    X_escalado, scaler = escalar_features(df, colunas)

    modelo_kmeans = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    df = df.copy()
    df["cluster"] = modelo_kmeans.fit_predict(X_escalado)
    return df, modelo_kmeans, scaler


# ATENÇÃO: k=5 é só o valor de partida usado no pipeline original — revalidar
# contra o gráfico de cotovelo/silhouette acima antes de tratar como definitivo.
K_ESCOLHIDO = 5
df_clusterizado, modelo_kmeans, scaler_cluster = clusterizar_faixas(df_limpo, k=K_ESCOLHIDO)

In [ ]:
def perfil_dos_clusters(df: pd.DataFrame, colunas: list = None):
    """
    GQ: 'existe uma combinação de características associada às músicas de
    alta popularidade?' — aqui respondida por cluster: cada grupo tem uma
    "assinatura sonora" e uma taxa de hits associada.
    """
    colunas = colunas or FEATURES_NUMERICAS
    perfil = df.groupby("cluster").agg(
        n_faixas=("cluster", "size"),
        taxa_de_hits=("is_hit", "mean"),
        popularidade_media=(COLUNAS["popularidade"], "mean"),
        **{f"{c}_medio": (c, "mean") for c in colunas},
    ).sort_values("taxa_de_hits", ascending=False)
    return perfil


perfil_dos_clusters(df_clusterizado)

In [ ]:
def visualizar_clusters_2d(df: pd.DataFrame, colunas: list = None):
    """Projeta os clusters em 2D via PCA só para inspeção visual."""
    colunas = colunas or FEATURES_NUMERICAS
    X_escalado, _ = escalar_features(df, colunas)
    coords = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(X_escalado)

    plt.figure(figsize=(8, 6))
    scatter = plt.scatter(coords[:, 0], coords[:, 1], c=df["cluster"],
                           cmap="tab10", alpha=0.5, s=10)
    plt.title("Clusters de faixas (projeção PCA 2D)")
    plt.xlabel("Componente 1")
    plt.ylabel("Componente 2")
    plt.colorbar(scatter, label="Cluster")
    plt.tight_layout()


visualizar_clusters_2d(df_clusterizado)

## Artefatos de saída

Salva o dataframe clusterizado (com `is_hit` e `cluster`) e o modelo/scaler treinados para o módulo 05 consumir sem reexecutar treino.

In [ ]:
import pickle

df_clusterizado.to_csv(ARTIFACTS_DIR / "df_clusterizado.csv", index=False)
with open(ARTIFACTS_DIR / "modelo_rf.pkl", "wb") as f:
    pickle.dump(modelo, f)
with open(ARTIFACTS_DIR / "scaler_cluster.pkl", "wb") as f:
    pickle.dump(scaler_cluster, f)

print("Artefatos salvos em:", ARTIFACTS_DIR.resolve())

## Síntese da ideação

- **Rótulo de sucesso:** `is_hit` = top 20% de `popularity` (threshold global — decisão em aberto, ver módulo 02).
- **EDA:** `comparar_hits_vs_demais` aponta quais features de áudio mais distinguem hits; `densidade_e_volatilidade_por_genero` aponta em quais gêneros vale mais a pena procurar oportunidades (alta densidade de hits) ou tomar cuidado (alta volatilidade).
- **Modelo:** Random Forest balanceado por classe, avaliado por classification report + ROC-AUC; `feature_importances_` traduz o modelo em "o que pesa mais para ser hit".
- **Clustering:** KMeans sobre as features escalonadas descreve arquétipos sonoros e a taxa de hits de cada um — visão complementar (não probabilística) ao modelo supervisionado.

Isso responde diretamente às GQs das Frentes 2 e 3. O próximo módulo usa esses resultados (modelo, clusters, perfil de hits) para prototipar as funcionalidades voltadas ao usuário final — [05 · Protótipo do Produto](05_prototipo_produto.ipynb).